In [ ]:
# Research Assistant Chatbot with Groq API
# Install required packages
!pip install -q groq gradio PyPDF2 python-docx requests beautifulsoup4

import os
import gradio as gr
from groq import Groq
import PyPDF2
import docx
import io
import re
from typing import List, Tuple
import requests
from bs4 import BeautifulSoup

# Initialize Groq client
def init_groq(api_key: str):
    return Groq(api_key=api_key)

# Tab 1: Ask Questions
def ask_questions(question: str, history: List, api_key: str, model: str):
    if not api_key:
        return history + [("Error", "Please enter your Groq API key first.")], history

    try:
        client = init_groq(api_key)

        # Build conversation history
        messages = [{"role": "system", "content": "You are a helpful research assistant. Provide detailed, accurate, and well-structured answers."}]
        for user_msg, assistant_msg in history:
            if user_msg:
                messages.append({"role": "user", "content": user_msg})
            if assistant_msg:
                messages.append({"role": "assistant", "content": assistant_msg})

        messages.append({"role": "user", "content": question})

        # Get response from Groq
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=0.7,
            max_tokens=2000
        )

        answer = response.choices[0].message.content
        history.append((question, answer))

        return history, history

    except Exception as e:
        return history + [(question, f"Error: {str(e)}")], history

# Tab 2: Summarize Text
def summarize_text(text: str, file, api_key: str, model: str, summary_type: str):
    if not api_key:
        return "Please enter your Groq API key first."

    try:
        # Extract text from file if provided - file is a file path string
        if file is not None:
            if file.endswith('.pdf'):
                with open(file, 'rb') as f:
                    pdf_reader = PyPDF2.PdfReader(f)
                    text = ""
                    for page in pdf_reader.pages:
                        text += page.extract_text()
            elif file.endswith('.docx'):
                doc = docx.Document(file)
                text = "\n".join([paragraph.text for paragraph in doc.paragraphs])
            elif file.endswith('.txt'):
                with open(file, 'r', encoding='utf-8') as f:
                    text = f.read()

        if not text:
            return "No text provided for summarization."

        # Limit text to approximately 20,000 characters (~5000 tokens) to avoid rate limits
        max_chars = 20000
        if len(text) > max_chars:
            text = text[:max_chars]
            truncation_note = f"\n\n[Note: Text was truncated to {max_chars} characters due to length. Processing first portion only.]"
        else:
            truncation_note = ""

        client = init_groq(api_key)

        summary_prompts = {
            "Brief": "Provide a brief 2-3 sentence summary of the following text:",
            "Detailed": "Provide a detailed summary with key points and important details from the following text:",
            "Bullet Points": "Summarize the following text using bullet points to highlight main ideas:"
        }

        prompt = f"{summary_prompts[summary_type]}\n\n{text}"

        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are an expert at summarizing texts accurately and concisely."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.5,
            max_tokens=1500
        )

        return response.choices[0].message.content + truncation_note

    except Exception as e:
        error_msg = str(e)
        if "rate_limit" in error_msg.lower() or "tpm" in error_msg.lower():
            return "⚠️ Rate limit exceeded. Please wait a moment and try again, or try with a shorter document/text."
        return f"Error: {error_msg}"

# Tab 3: PDF Assistant
def pdf_assistant(pdf_file, question: str, api_key: str, model: str):
    if not api_key:
        return "Please enter your Groq API key first."

    if not pdf_file:
        return "Please upload a PDF file."

    try:
        # Extract text from PDF - pdf_file is a file path string
        with open(pdf_file, 'rb') as f:
            pdf_reader = PyPDF2.PdfReader(f)
            pdf_text = ""
            for page in pdf_reader.pages:
                pdf_text += page.extract_text()

        if not pdf_text:
            return "Could not extract text from PDF."

        # Limit text to avoid rate limits
        max_chars = 15000
        if len(pdf_text) > max_chars:
            pdf_text = pdf_text[:max_chars]
            truncation_note = "\n\n[Note: PDF content was truncated due to length. Showing analysis of first portion only.]"
        else:
            truncation_note = ""

        client = init_groq(api_key)

        prompt = f"Based on the following document, answer this question: {question}\n\nDocument:\n{pdf_text}"

        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are a PDF analysis assistant. Answer questions based on the provided document content."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.5,
            max_tokens=1500
        )

        return response.choices[0].message.content + truncation_note

    except Exception as e:
        error_msg = str(e)
        if "rate_limit" in error_msg.lower() or "tpm" in error_msg.lower():
            return "⚠️ Rate limit exceeded. Please wait a moment and try again with your question."
        return f"Error: {error_msg}"

# Tab 4: Literature Recommendations
def literature_recommendations(topic: str, num_recommendations: int, api_key: str, model: str):
    if not api_key:
        return "Please enter your Groq API key first."

    try:
        client = init_groq(api_key)

        prompt = f"Recommend {num_recommendations} academic papers, books, or articles about '{topic}'. For each recommendation, provide:\n1. Title\n2. Authors\n3. Brief description\n4. Why it's relevant\n\nFormat as a numbered list."

        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are an expert research librarian who recommends high-quality academic literature."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.7,
            max_tokens=2500
        )

        return response.choices[0].message.content

    except Exception as e:
        return f"Error: {str(e)}"

# Tab 5: Code Generator
def generate_code(language: str, description: str, api_key: str, model: str):
    if not api_key:
        return "Please enter your Groq API key first."

    try:
        client = init_groq(api_key)

        prompt = f"Generate {language} code for the following task:\n{description}\n\nProvide clean, well-commented code with explanations."

        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are an expert programmer. Generate clean, efficient, and well-documented code."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.5,
            max_tokens=2500
        )

        return response.choices[0].message.content

    except Exception as e:
        return f"Error: {str(e)}"

# Create Gradio Interface
def create_interface():
    with gr.Blocks(title="Research Assistant Chatbot", theme=gr.themes.Soft()) as demo:
        gr.Markdown("# 🤖 Research Assistant Chatbot with Groq API")
        gr.Markdown("A comprehensive AI assistant for research, document analysis, and code generation.")

        with gr.Row():
            api_key_input = gr.Textbox(
                label="Groq API Key",
                placeholder="Enter your Groq API key here...",
                type="password"
            )
            model_dropdown = gr.Dropdown(
                choices=["llama-3.3-70b-versatile", "llama-3.1-70b-versatile", "mixtral-8x7b-32768"],
                value="llama-3.3-70b-versatile",
                label="Model"
            )

        with gr.Tabs():
            # Tab 1: Ask Questions
            with gr.Tab("💬 Ask Questions"):
                chatbot = gr.Chatbot(label="Chat History", height=400)
                with gr.Row():
                    question_input = gr.Textbox(
                        label="Your Question",
                        placeholder="Ask me anything...",
                        lines=2
                    )
                with gr.Row():
                    ask_btn = gr.Button("Send", variant="primary")
                    clear_btn = gr.Button("Clear Chat")

                chat_state = gr.State([])

                ask_btn.click(
                    ask_questions,
                    inputs=[question_input, chat_state, api_key_input, model_dropdown],
                    outputs=[chatbot, chat_state]
                ).then(lambda: "", None, question_input)

                clear_btn.click(lambda: ([], []), None, [chatbot, chat_state])

            # Tab 2: Summarize Text
            with gr.Tab("📄 Summarize Text"):
                with gr.Row():
                    with gr.Column():
                        text_input = gr.Textbox(
                            label="Text to Summarize",
                            placeholder="Paste your text here or upload a file...",
                            lines=10
                        )
                        file_input = gr.File(label="Or Upload File (PDF, DOCX, TXT)")
                        summary_type = gr.Radio(
                            choices=["Brief", "Detailed", "Bullet Points"],
                            value="Detailed",
                            label="Summary Type"
                        )
                        summarize_btn = gr.Button("Summarize", variant="primary")

                    with gr.Column():
                        summary_output = gr.Textbox(
                            label="Summary",
                            lines=15
                        )

                summarize_btn.click(
                    summarize_text,
                    inputs=[text_input, file_input, api_key_input, model_dropdown, summary_type],
                    outputs=summary_output
                )

            # Tab 3: PDF Assistant
            with gr.Tab("📕 PDF Assistant"):
                with gr.Row():
                    with gr.Column():
                        pdf_upload = gr.File(label="Upload PDF", file_types=[".pdf"])
                        pdf_question = gr.Textbox(
                            label="Question about the PDF",
                            placeholder="What would you like to know about this document?",
                            lines=3
                        )
                        pdf_btn = gr.Button("Ask", variant="primary")

                    with gr.Column():
                        pdf_output = gr.Textbox(
                            label="Answer",
                            lines=15
                        )

                pdf_btn.click(
                    pdf_assistant,
                    inputs=[pdf_upload, pdf_question, api_key_input, model_dropdown],
                    outputs=pdf_output
                )

            # Tab 4: Literature Recommendations
            with gr.Tab("📚 Literature Recommendations"):
                with gr.Row():
                    with gr.Column():
                        topic_input = gr.Textbox(
                            label="Research Topic",
                            placeholder="Enter your research topic...",
                            lines=2
                        )
                        num_recs = gr.Slider(
                            minimum=1,
                            maximum=10,
                            value=5,
                            step=1,
                            label="Number of Recommendations"
                        )
                        lit_btn = gr.Button("Get Recommendations", variant="primary")

                    with gr.Column():
                        lit_output = gr.Textbox(
                            label="Recommendations",
                            lines=15
                        )

                lit_btn.click(
                    literature_recommendations,
                    inputs=[topic_input, num_recs, api_key_input, model_dropdown],
                    outputs=lit_output
                )

            # Tab 5: Code Generator
            with gr.Tab("💻 Code Generator"):
                with gr.Row():
                    with gr.Column():
                        language_input = gr.Dropdown(
                            choices=["Python", "Java", "JavaScript", "C++", "C", "R", "SQL", "HTML/CSS"],
                            value="Python",
                            label="Programming Language"
                        )
                        code_description = gr.Textbox(
                            label="Describe the code you want",
                            placeholder="E.g., How to get the maximum number out of 5 given numbers?",
                            lines=5
                        )
                        code_btn = gr.Button("Generate Code", variant="primary")

                    with gr.Column():
                        code_output = gr.Code(
                            label="Generated Code",
                            lines=15
                        )

                code_btn.click(
                    generate_code,
                    inputs=[language_input, code_description, api_key_input, model_dropdown],
                    outputs=code_output
                )

            # Tab 6: Info
            with gr.Tab("ℹ️ Info"):
                gr.Markdown("""
                ## About This Research Assistant

                This chatbot provides multiple AI-powered features for research and development:

                ### Features:
                - **Ask Questions**: General-purpose Q&A with conversation history
                - **Summarize Text**: Generate summaries from text or uploaded documents
                - **PDF Assistant**: Upload PDFs and ask questions about their content
                - **Literature Recommendations**: Get curated research paper and book suggestions
                - **Code Generator**: Generate code in multiple programming languages

                ### How to Use:
                1. Enter your **Groq API Key** at the top (get one free at [console.groq.com](https://console.groq.com))
                2. Select your preferred model
                3. Navigate to any tab and start using the features

                ### Available Models:
                - **llama-3.3-70b-versatile**: Latest and most capable
                - **llama-3.1-70b-versatile**: Balanced performance
                - **mixtral-8x7b-32768**: Large context window

                ### Tips:
                - Be specific in your questions for better results
                - For PDF analysis, ensure the PDF contains extractable text
                - Code generation works best with clear, detailed descriptions

                Built with Groq API and Gradio
                """)

        gr.Markdown("---")
        gr.Markdown("*Powered by Groq's lightning-fast LLM inference*")

    return demo

if __name__ == "__main__":
    demo = create_interface()
    demo.launch(share=True, debug=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 4.8 MB/s eta 0:00:00


/tmp/ipykernel_1483/926298198.py:215: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Research Assistant Chatbot", theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_1483/926298198.py:234: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="Chat History", height=400)
/tmp/ipykernel_1483/926298198.py:234: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="Chat History", height=400)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://03162667ebdc9ad3cd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
